<a href="https://colab.research.google.com/github/maliah1010/OpenVocal/blob/experiments%2Fmalia/Edge_Native_TTS_Prototype_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01: Edge-Native TTS Baseline (Proof of Concept)

**Author:** Malia Hosseini
**Date:** March 2026

### Objective
To establish a working baseline inference pipeline using an ultra-lightweight, edge-native decoder architecture. This notebook serves as the initial Proof of Concept (PoC) for our custom text-to-speech engine.

### Technical Specs
* **Model Context:** Kokoro-82M (Decoder-only)
* **License:** Apache 2.0
* **Hardware:** NVIDIA T4 GPU (Google Colab Free Tier)
* **Goal:** Validate baseline acoustic naturalness and verify the environment setup for future fine-tuning and deterministic frame-level steering.

## Step 1: Environment Setup & Dependencies
Before we initialize the model, we need to install our core libraries and system dependencies:

* **`espeak-ng`**: An open-source system-level synthesizer. We are using this as our grapheme-to-phoneme backend to accurately convert English text into raw phonetic data.
* **`kokoro`**: The core Python package for the ultra-lightweight Kokoro-82M model.
* **`soundfile`**: A necessary library to read, write, and process the raw audio arrays generated by the model.

In [1]:
# CELL 1: ENVIRONMENT SETUP
!apt-get -qq -y install espeak-ng > /dev/null 2>&1
!pip install -q kokoro soundfile

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.3/48.3 kB 2.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 99.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.7/82.7 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 237.9/237.9 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 734.0/734.0 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.4/213.4 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 28.7 MB/s eta 0:00:00


## Step 2: GPU Verification & Model Initialization
In this step, we initialize our PyTorch environment and load the ultra-lightweight Kokoro-82M model weights into memory.

* **Hardware Check:** We first verify that Google Colab has successfully assigned us an NVIDIA GPU (`cuda`). Running inference on a GPU is what allows us to hit our target sub-200ms latency.
* **Pipeline Initialization:** We instantiate the `KPipeline` for American English (`lang_code='a'`). Because this is an 82-million parameter decoder-only model, downloading the weights takes only a few seconds and occupies roughly 2GB of VRAM, making it perfectly edge-native.

In [2]:
# CELL 2: INITIALIZE THE MODEL
from kokoro import KPipeline
import torch

# 1. Verify we are using the Colab GPU (CUDA) and not the slow CPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# 2. Initialize the TTS Pipeline
print("Downloading model weights... this may take a few seconds.")
pipeline = KPipeline(lang_code='a') # 'a' for American English

print("Kokoro model initialized and ready!")

Using device: cpu


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:990: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


kokoro-v1_0.pth:   0%|          | 0.00/327M [00:00<?, ?B/s]

Kokoro model initialized and ready!


## Step 3: Inference & Acoustic Rendering
This is the core generation loop. We pass a raw text string through the initialized model to synthesize our first speech sample.

* **Linguistic Front-End:** The model first converts our text into discrete phonemes. We will print these out to verify that our grapheme-to-phoneme alignment is functioning correctly.
* **Audio Generation:** The neural network decodes those phonemes into a continuous audio array at a 24,000 Hz sample rate.
* **Output:** We use Colab's native `IPython.display` to render a playable audio widget directly in the browser so we can evaluate the baseline acoustic naturalness.

In [3]:
# CELL 3: GENERATE AND PLAY AUDIO
from IPython.display import display, Audio

# 1. Define the text we want to synthesize
text = "Hello Malia. This is the first successful test of our edge-native text-to-speech engine. The environment is perfectly set up and ready for our working session."

# 2. Generate the audio
# 'af_heart' is a default high-quality American Female voice.
print("Synthesizing audio...")
generator = pipeline(
    text,
    voice='af_heart', # You can also try 'am_michael' for a male voice
    speed=1.0,
    split_pattern=r'\n+'
)

# 3. Extract and play the generated audio
for i, (graphemes, phonemes, audio) in enumerate(generator):
    print("Audio generated successfully!")
    print(f"Phonemes used: {phonemes}")

    # Render the playable widget (Kokoro uses a 24,000 Hz sample rate)
    display(Audio(data=audio, rate=24000))

Synthesizing audio...


voices/af_heart.pt:   0%|          | 0.00/523k [00:00<?, ?B/s]

Audio generated successfully!
Phonemes used: həlˈO mˈAliə. ðˌɪs ɪz ðə fˈɜɹst səksˈɛsfᵊl tˈɛst ʌv ˌWəɹ ˌɛʤnˈATɪv tˈɛksttəspˈiʧ ˈɛnʤən. ði ənvˈIɹənmᵊnt ɪz pˈɜɹfəktli sˈɛt ˌʌp ænd ɹˈɛdi fɔɹ ˌWəɹ wˈɜɹkɪŋ sˈɛʃən.
